# CAMELS-US clean HPO → selection → ensemble → test evaluation notebook

Clean GitHub-ready consolidation of the minimum reproducibility chain:

1. Define the hyperparameter search space.
2. Generate random-search NeuralHydrology YAML configs.
3. Run HPO training by random search.
4. Extract validation metrics.
5. Select best regional Top-10 and cluster-wise configs.
6. Rebuild selected configs with the final 10 random seeds.
7. Run final training and test evaluation.
8. Extract test predictions.
9. Build median ensembles.
10. Evaluate ensembles on test data.

The plotting/archive notebooks are intentionally excluded.

## 0. Provenance

Consolidated from:

- `Master_Hypertune_ver04_updated_Jul2025.ipynb`
- `Master_Hypertune_ver03_Nov.2024.ipynb`
- `Master_Experiment_Feb2026_PredictionsTimeseries.ipynb`
- `Master_Experiment_Feb2026_GeneratingMetrics.ipynb`
- `metrics.py`

The figure notebooks are downstream visualization code and are not needed for the reproducibility workflow.

## 1. Required folder layout

Recommended structure:

```text
CAMELS_US_repro/
├─ configs/
│  └─ configRegional_Fred.yml
├─ data/
│  ├─ CAMELS_US/
│  ├─ 531Basins.csv
│  └─ camels_attributes_v2.0/
│     └─ attributes_cluster_splits/
│        ├─ cluster_0.txt
│        ├─ cluster_1.txt
│        ├─ cluster_2.txt
│        ├─ cluster_3.txt
│        ├─ cluster_4.txt
│        └─ cluster_5.txt
├─ runs/
│  ├─ hpo_random_search/
│  ├─ final_top10/
│  └─ final_clusterwise/
├─ outputs/
│  ├─ evaluations/
│  └─ pickles/
└─ metrics.py
```

You need the NeuralHydrology-compatible CAMELS-US data, the original base YAML config, the 531-basin list, cluster split files, and `metrics.py`.

In [ ]:
from __future__ import annotations

import os
import re
import yaml
import pickle
import logging
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.model_selection import ParameterSampler

LOGGER = logging.getLogger("camels_us_clean_hpo")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

try:
    from neuralhydrology.nh_run import Config
    from neuralhydrology.nh_run_scheduler import schedule_runs
    from neuralhydrology.evaluation import get_tester
    NEURALHYDROLOGY_AVAILABLE = True
except Exception as e:
    NEURALHYDROLOGY_AVAILABLE = False
    LOGGER.warning("NeuralHydrology is not available in this environment. Training/evaluation cells will not run here. Error: %r", e)

## 2. User configuration

Edit this cell before running.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

CONFIG_DIR = PROJECT_ROOT / "configs"
DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

BASE_CONFIG_FILE = CONFIG_DIR / "configRegional_Fred.yml"

HPO_CONFIG_DIR = CONFIG_DIR / "LSTM_config_Hypertuning_files"
HPO_RUNS_DIR = RUNS_DIR / "hpo_random_search"

SELECTED_CONFIG_DIR = CONFIG_DIR / "selected_final_configs"
FINAL_TOP10_CONFIG_DIR = SELECTED_CONFIG_DIR / "top10"
FINAL_CLUSTER_CONFIG_DIR = SELECTED_CONFIG_DIR / "clusterwise"

FINAL_TOP10_RUNS_DIR = RUNS_DIR / "final_top10"
FINAL_CLUSTER_RUNS_DIR = RUNS_DIR / "final_clusterwise"

EVAL_DIR = OUTPUT_DIR / "evaluations"
PICKLE_DIR = OUTPUT_DIR / "pickles"

for d in [HPO_CONFIG_DIR, HPO_RUNS_DIR, FINAL_TOP10_CONFIG_DIR, FINAL_CLUSTER_CONFIG_DIR,
          FINAL_TOP10_RUNS_DIR, FINAL_CLUSTER_RUNS_DIR, EVAL_DIR, PICKLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASINS_FILE = DATA_DIR / "531Basins.csv"
ATTRIBUTES_DIR = DATA_DIR / "camels_attributes_v2.0"
CLUSTER_SPLIT_DIR = ATTRIBUTES_DIR / "attributes_cluster_splits"

N_RANDOM_CONFIGS = 2000
RANDOM_STATE = 42  # original exploratory code did not fix this; use None to mimic non-deterministic sampling

VALIDATION_EPOCHS = [20, 25, 30]
DEFAULT_VALIDATION_EPOCH = 30
FINAL_TEST_EPOCH = None  # None = auto-detect latest/only model_epoch folder under test/

SEED_VALUES = [191889, 307443, 328237, 385445, 473461, 643969, 665980, 845563, 985957, 993061]

SELECTION_METRIC = "SFmean_NSE"
N_TOP_GLOBAL = 10
N_TOP_PER_CLUSTER = 3
N_CLUSTERS = 6

FREQ = "1D"
SIM_KEY = "QObs(mm/d)_sim"
OBS_KEY = "QObs(mm/d)_obs"

## 3. Hyperparameter space used for random search

In [ ]:
PARAM_GRID = {
    "hidden_size": [16, 32, 64, 128, 256],
    "initial_forget_bias": [-3, -1, 0, 1, 3, 5],
    "output_dropout": [0, 0.2, 0.4],
    "batch_size": [32, 64, 128, 256, 512],
    "learning_rate": [
        {0: lr_1, 10: lr_2, 25: lr_3}
        for lr_1 in [1e-3, 1e-2]
        for lr_2 in [5e-4, 1e-3, 5e-3]
        for lr_3 in [1e-4, 1e-3]
    ],
    "target_noise_std": [0, 0.01, 0.02, 0.05, 0.1],
    "loss": ["NSE", "RMSE"],
    "seq_length": [90, 120, 180, 270, 365, 450, 540, 730, 900, 1095, 1460, 1825],
}

HP_COLUMNS = [
    "seq_length", "batch_size", "Lr0", "Lr10", "Lr25", "loss", "hidden_size",
    "output_dropout", "initial_forget_bias", "target_noise_std", "seed", "Model"
]

METRIC_RENAME = {
    "NSE": "SFmean_NSE",
    "KGE": "SFmean_KGE",
    "MSE": "SFmean_MSE",
    "RMSE": "SFmean_RMSE",
    "Alpha-NSE": "SFmean_Alpha-NSE",
    "Beta-NSE": "SFmean_Beta-NSE",
    "Beta-KGE": "SFmean_Beta-KGE",
    "Pearson-r": "SFmean_Pearson-r",
    "FHV": "SFmean_FHV",
    "FMS": "SFmean_FMS",
    "FLV": "SFmean_FLV",
    "Peak-Timing": "SFmean_Peak-Timing",
    "Peak-MAPE": "SFmean_Peak-MAPE",
    "Missed-Peaks": "SFmean_Missed-Peaks",
}

## 4. Generate random-search YAML configs

In [ ]:
def load_yaml(path: Path) -> Dict[str, Any]:
    with Path(path).open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def save_yaml(obj: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with Path(path).open("w", encoding="utf-8") as f:
        yaml.safe_dump(dict(obj), f, sort_keys=False)

def generate_random_search_configs(
    base_config_file: Path,
    output_dir: Path,
    param_grid: Mapping[str, Sequence[Any]],
    n_samples: int,
    random_state: Optional[int] = 42,
    prefix: str = "LSTM_config_Hypertuning",
    overwrite: bool = False,
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)
    if overwrite:
        for p in output_dir.glob("*.yml"):
            p.unlink()

    base_config = load_yaml(base_config_file)
    sampler = ParameterSampler(param_grid, n_iter=n_samples, random_state=random_state)

    records = []
    for i, params in enumerate(sampler):
        config = dict(base_config)
        config.update(params)
        experiment_name = config.get("experiment_name", "CAMELSUS_RS")
        config["experiment_name"] = f"{experiment_name}_{i:04d}"

        out_file = output_dir / f"{prefix}_{i:04d}.yml"
        save_yaml(config, out_file)

        lr = params.get("learning_rate", {})
        records.append({
            "config_file": str(out_file),
            "Model": config["experiment_name"],
            "Lr0": lr.get(0), "Lr10": lr.get(10), "Lr25": lr.get(25),
            **{k: v for k, v in params.items() if k != "learning_rate"},
        })

    df = pd.DataFrame(records)
    df.to_csv(EVAL_DIR / "generated_random_search_configs.csv", index=False)
    LOGGER.info("Generated %d configs in %s", len(df), output_dir)
    return df

# generated_configs = generate_random_search_configs(
#     BASE_CONFIG_FILE, HPO_CONFIG_DIR, PARAM_GRID, N_RANDOM_CONFIGS, random_state=RANDOM_STATE
# )

## 5. Run HPO training by random search

In [ ]:
def run_neuralhydrology_schedule(config_dir: Path, gpu_ids: Sequence[int] = (0,), runs_per_gpu: int = 1, mode: str = "train") -> None:
    if not NEURALHYDROLOGY_AVAILABLE:
        raise ImportError("NeuralHydrology is not available. Install/activate it before running this cell.")
    schedule_runs(mode=mode, directory=Path(config_dir), gpu_ids=list(gpu_ids), runs_per_gpu=runs_per_gpu)

# Example:
# run_neuralhydrology_schedule(HPO_CONFIG_DIR, gpu_ids=[0, 1], runs_per_gpu=5, mode="train")

## 6. Extract validation/test metrics from NeuralHydrology run folders

In [ ]:
def normalize_model_name(name: str) -> str:
    return str(name).replace("531_basins_Train_100RandomSearches_", "")

def find_epoch_result_file(run_dir: Path, period: str, epoch: Optional[int] = None) -> Optional[Path]:
    period_dir = Path(run_dir) / period
    if not period_dir.exists():
        return None
    if epoch is not None:
        exact = period_dir / f"model_epoch{epoch:03d}" / f"{period}_results.p"
        if exact.exists():
            return exact
    candidates = sorted(period_dir.glob(f"model_epoch*/{period}_results.p"))
    if not candidates:
        return None
    def epoch_number(p: Path) -> int:
        m = re.search(r"model_epoch(\d+)", str(p))
        return int(m.group(1)) if m else -1
    return sorted(candidates, key=epoch_number)[-1]

def extract_hyperparameters_from_config(config_path: Path, params: Iterable[str]) -> Dict[str, Any]:
    cfg = load_yaml(config_path)
    out = {}
    for p in params:
        if p not in cfg:
            continue
        if p == "learning_rate":
            lr = cfg[p] or {}
            out["Lr0"] = lr.get(0) or lr.get("0")
            out["Lr10"] = lr.get(10) or lr.get("10")
            out["Lr25"] = lr.get(25) or lr.get("25")
        else:
            out[p] = cfg[p]
    out["Model"] = normalize_model_name(cfg.get("experiment_name", config_path.parent.name))
    return out

def flatten_nh_metrics(results: Mapping[str, Any], model_name: str, epoch: int) -> pd.DataFrame:
    rows = []
    for basin, freq_metrics in results.items():
        for freq, metric_values in freq_metrics.items():
            metrics_dict = {k: v for k, v in metric_values.items() if k != "xr"}
            rows.append({"basin": str(basin), "freq": freq, "epoch": epoch, "Model": model_name, **metrics_dict})
    return pd.DataFrame(rows)

def collect_period_metrics_from_runs(
    parent_dir: Path,
    output_csv: Path,
    period: str = "validation",
    epochs: Sequence[int] = (30,),
    params: Iterable[str] = ("hidden_size", "initial_forget_bias", "output_dropout", "batch_size",
                             "learning_rate", "target_noise_std", "seq_length", "loss", "seed"),
) -> pd.DataFrame:
    hp_records, metric_frames = [], []
    run_dirs = [p for p in Path(parent_dir).iterdir() if p.is_dir()]
    LOGGER.info("Scanning %d run folders in %s", len(run_dirs), parent_dir)

    for run_dir in run_dirs:
        config_path = run_dir / "config.yml"
        if not config_path.exists():
            continue
        hp = extract_hyperparameters_from_config(config_path, params=params)
        model_name = hp["Model"]
        hp_records.append(hp)

        for epoch in epochs:
            result_path = find_epoch_result_file(run_dir, period=period, epoch=epoch)
            if result_path is None:
                LOGGER.warning("No %s result found for %s epoch=%s", period, run_dir.name, epoch)
                continue
            with result_path.open("rb") as f:
                results = pickle.load(f)
            metric_frames.append(flatten_nh_metrics(results, model_name=model_name, epoch=epoch))

    if not metric_frames:
        raise FileNotFoundError(f"No {period} result files found in {parent_dir}")

    df_h = pd.DataFrame(hp_records).drop_duplicates(subset=["Model"])
    df_v = pd.concat(metric_frames, ignore_index=True)
    df = pd.merge(df_h, df_v, on="Model", how="right")
    df["Model"] = df["Model"].map(normalize_model_name)
    df = df.rename(columns=METRIC_RENAME)

    front_cols = [c for c in HP_COLUMNS + ["basin", "freq", "epoch"] if c in df.columns]
    other_cols = [c for c in df.columns if c not in front_cols]
    df = df[front_cols + other_cols]

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    LOGGER.info("Saved %s metrics to %s with shape %s", period, output_csv, df.shape)
    return df

# validation_df = collect_period_metrics_from_runs(
#     HPO_RUNS_DIR, EVAL_DIR / "Ver_results_2000RS_validation_epochs_20_25_30.csv",
#     period="validation", epochs=VALIDATION_EPOCHS
# )

## 7. Select best regional Top-10 and cluster-wise models

In [ ]:
def load_cluster_splits(cluster_dir: Path, n_clusters: int = 6) -> Dict[int, List[str]]:
    clusters = {}
    for i in range(n_clusters):
        p = cluster_dir / f"cluster_{i}.txt"
        with p.open("r", encoding="utf-8") as f:
            clusters[i] = [line.strip() for line in f if line.strip()]
    return clusters

def summarize_validation(
    df: pd.DataFrame,
    metric: str = SELECTION_METRIC,
    aggregate: str = "mean",
    group_cols: Sequence[str] = ("Model", "epoch"),
) -> pd.DataFrame:
    if aggregate == "mean":
        s = df.groupby(list(group_cols), dropna=False)[metric].mean()
    elif aggregate == "median":
        s = df.groupby(list(group_cols), dropna=False)[metric].median()
    elif aggregate == "max":
        s = df.groupby(list(group_cols), dropna=False)[metric].max()
    else:
        raise ValueError("aggregate must be mean, median, or max.")
    out = s.reset_index().rename(columns={metric: f"{metric}_{aggregate}"})
    return out.sort_values(f"{metric}_{aggregate}", ascending=False)

def select_top_global_models(validation_df: pd.DataFrame, n_top: int = N_TOP_GLOBAL,
                             metric: str = SELECTION_METRIC, aggregate: str = "mean") -> pd.DataFrame:
    summary = summarize_validation(validation_df, metric=metric, aggregate=aggregate)
    score_col = f"{metric}_{aggregate}"
    selected = summary.sort_values(score_col, ascending=False).drop_duplicates("Model", keep="first").head(n_top).copy()
    selected["selection_family"] = "Top10"
    selected["selection_rank"] = np.arange(1, len(selected) + 1)
    return selected

def select_top_models_per_cluster(validation_df: pd.DataFrame, cluster_splits: Mapping[int, Sequence[str]],
                                  n_top_per_cluster: int = N_TOP_PER_CLUSTER,
                                  metric: str = SELECTION_METRIC, aggregate: str = "mean") -> pd.DataFrame:
    frames = []
    score_col = f"{metric}_{aggregate}"
    for cluster_id, basins in cluster_splits.items():
        sub = validation_df[validation_df["basin"].astype(str).isin(set(map(str, basins)))].copy()
        if sub.empty:
            LOGGER.warning("Cluster %s has no matching validation rows.", cluster_id)
            continue
        summary = summarize_validation(sub, metric=metric, aggregate=aggregate)
        selected = summary.sort_values(score_col, ascending=False).drop_duplicates("Model", keep="first").head(n_top_per_cluster).copy()
        selected["cluster"] = cluster_id
        selected["selection_family"] = "ClusterWise"
        selected["selection_rank"] = np.arange(1, len(selected) + 1)
        frames.append(selected)
    return pd.concat(frames, ignore_index=True)

# validation_df = pd.read_csv(EVAL_DIR / "Ver_results_2000RS_validation_epochs_20_25_30.csv", dtype={"basin": str})
# selected_top10 = select_top_global_models(validation_df, aggregate="mean")
# clusters = load_cluster_splits(CLUSTER_SPLIT_DIR, n_clusters=N_CLUSTERS)
# selected_clusterwise = select_top_models_per_cluster(validation_df, clusters, aggregate="mean")
# selected_top10.to_csv(EVAL_DIR / "selected_top10_models.csv", index=False)
# selected_clusterwise.to_csv(EVAL_DIR / "selected_clusterwise_models.csv", index=False)

## 8. Create final seeded configs for selected models

In [ ]:
def find_run_dir_by_model(parent_dir: Path, model_name: str) -> Optional[Path]:
    model_name = normalize_model_name(model_name)
    for p in Path(parent_dir).iterdir():
        if not p.is_dir():
            continue
        cfg_path = p / "config.yml"
        if not cfg_path.exists():
            continue
        try:
            exp = normalize_model_name(load_yaml(cfg_path).get("experiment_name", p.name))
        except Exception:
            exp = normalize_model_name(p.name)
        if exp == model_name or normalize_model_name(p.name) == model_name:
            return p
    return None

def create_seeded_selected_configs(
    selected_df: pd.DataFrame,
    hpo_runs_dir: Path,
    output_dir: Path,
    seeds: Sequence[int] = SEED_VALUES,
    name_prefix: str = "Selected",
) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)
    records = []
    for _, row in selected_df.iterrows():
        model_name = normalize_model_name(row["Model"])
        run_dir = find_run_dir_by_model(hpo_runs_dir, model_name)
        if run_dir is None:
            LOGGER.warning("Could not find HPO run folder for selected model %s", model_name)
            continue
        base_cfg = load_yaml(run_dir / "config.yml")
        selected_epoch = int(row["epoch"]) if "epoch" in row and not pd.isna(row["epoch"]) else DEFAULT_VALIDATION_EPOCH

        for seed in seeds:
            cfg = dict(base_cfg)
            cfg["seed"] = int(seed)
            cfg["experiment_name"] = f"{name_prefix}_{model_name}_epoch{selected_epoch:03d}_RS{seed}"
            out_file = output_dir / f"{cfg['experiment_name']}.yml"
            save_yaml(cfg, out_file)
            records.append({
                "source_model": model_name,
                "selected_epoch": selected_epoch,
                "seed": seed,
                "config_file": str(out_file),
                "experiment_name": cfg["experiment_name"],
            })
    manifest = pd.DataFrame(records)
    manifest.to_csv(output_dir / "manifest_selected_seeded_configs.csv", index=False)
    LOGGER.info("Created %d seeded selected configs in %s", len(manifest), output_dir)
    return manifest

# top10_manifest = create_seeded_selected_configs(selected_top10, HPO_RUNS_DIR, FINAL_TOP10_CONFIG_DIR, name_prefix="Top10")
# cluster_manifest = create_seeded_selected_configs(selected_clusterwise, HPO_RUNS_DIR, FINAL_CLUSTER_CONFIG_DIR, name_prefix="ClusterWise")

## 9. Train final selected models and evaluate them on test data

In [ ]:
# Train:
# run_neuralhydrology_schedule(FINAL_TOP10_CONFIG_DIR, gpu_ids=[0, 1], runs_per_gpu=5, mode="train")
# run_neuralhydrology_schedule(FINAL_CLUSTER_CONFIG_DIR, gpu_ids=[0, 1], runs_per_gpu=5, mode="train")

def evaluate_finished_runs_on_test(run_parent_dir: Path) -> None:
    if not NEURALHYDROLOGY_AVAILABLE:
        raise ImportError("NeuralHydrology is not available. Install/activate it before running this cell.")
    for run_dir in [p for p in Path(run_parent_dir).iterdir() if p.is_dir()]:
        cfg_path = run_dir / "config.yml"
        if not cfg_path.exists():
            continue
        cfg = Config(load_yaml(cfg_path))
        tester = get_tester(cfg=cfg, run_dir=run_dir, period="test", init_model=True)
        tester.evaluate(save_results=True, metrics=cfg.metrics)
        LOGGER.info("Test evaluation finished: %s", run_dir.name)

# evaluate_finished_runs_on_test(FINAL_TOP10_RUNS_DIR)
# evaluate_finished_runs_on_test(FINAL_CLUSTER_RUNS_DIR)

## 10. Extract test predictions and observations

In [ ]:
def read_seed_and_experiment(run_dir: Path) -> Tuple[str, str]:
    cfg = load_yaml(run_dir / "config.yml")
    return str(cfg.get("seed", "Unknown")), normalize_model_name(cfg.get("experiment_name", run_dir.name))

def squeeze_prediction_da(da: xr.DataArray) -> xr.DataArray:
    out = da
    if "time_step" in out.dims:
        out = out.sel(time_step=0)
    return out.squeeze(drop=True)

def extract_predictions_from_runs(
    run_parent_dir: Path,
    out_pickle: Path,
    freq: str = FREQ,
    sim_key: str = SIM_KEY,
    epoch: Optional[int] = FINAL_TEST_EPOCH,
) -> Dict[str, Dict[str, Dict[str, Dict[str, xr.DataArray]]]]:
    predictions = {}
    for run_dir in [p for p in Path(run_parent_dir).iterdir() if p.is_dir()]:
        result_file = find_epoch_result_file(run_dir, period="test", epoch=epoch)
        if result_file is None:
            LOGGER.warning("No test_results.p found for %s", run_dir.name)
            continue
        seed, model_name = read_seed_and_experiment(run_dir)
        with result_file.open("rb") as f:
            results = pickle.load(f)
        for basin, basin_dict in results.items():
            try:
                da = squeeze_prediction_da(basin_dict[freq]["xr"][sim_key])
            except Exception as e:
                LOGGER.warning("Missing %s/%s/%s in %s: %r", basin, freq, sim_key, run_dir.name, e)
                continue
            predictions.setdefault(seed, {}).setdefault(str(basin), {}).setdefault(sim_key, {})[model_name] = da
    out_pickle.parent.mkdir(parents=True, exist_ok=True)
    with out_pickle.open("wb") as f:
        pickle.dump(predictions, f, protocol=pickle.HIGHEST_PROTOCOL)
    return predictions

def extract_observations_from_runs(
    run_parent_dirs: Sequence[Path],
    out_pickle: Path,
    freq: str = FREQ,
    obs_key: str = OBS_KEY,
    epoch: Optional[int] = FINAL_TEST_EPOCH,
) -> Dict[str, xr.DataArray]:
    obs = {}
    for parent in run_parent_dirs:
        for run_dir in [p for p in Path(parent).iterdir() if p.is_dir()]:
            result_file = find_epoch_result_file(run_dir, period="test", epoch=epoch)
            if result_file is None:
                continue
            with result_file.open("rb") as f:
                results = pickle.load(f)
            for basin, basin_dict in results.items():
                if str(basin) in obs:
                    continue
                xr_ds = basin_dict.get(freq, {}).get("xr")
                if xr_ds is None:
                    continue
                key = obs_key if obs_key in xr_ds else next((k for k in xr_ds.keys() if str(k).endswith("_obs")), None)
                if key is not None:
                    obs[str(basin)] = squeeze_prediction_da(xr_ds[key])
    out_pickle.parent.mkdir(parents=True, exist_ok=True)
    with out_pickle.open("wb") as f:
        pickle.dump(obs, f, protocol=pickle.HIGHEST_PROTOCOL)
    return obs

# all_top10 = extract_predictions_from_runs(FINAL_TOP10_RUNS_DIR, PICKLE_DIR / "All_Top10_Configs_CAMELSUS_test.p")
# all_cluster = extract_predictions_from_runs(FINAL_CLUSTER_RUNS_DIR, PICKLE_DIR / "All_Cluster_wise_Configs_CAMELSUS_test.p")
# obs_test = extract_observations_from_runs([FINAL_TOP10_RUNS_DIR, FINAL_CLUSTER_RUNS_DIR], PICKLE_DIR / "Obs_test.p")

## 11. Build median ensembles

In [ ]:
def save_pickle(obj: Any, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(path: Path) -> Any:
    with path.open("rb") as f:
        return pickle.load(f)

def robust_agg(name: str) -> Callable[[np.ndarray, int], np.ndarray]:
    name = name.lower().strip()
    if name == "median":
        return lambda a, axis: np.nanmedian(a, axis=axis)
    if name == "mean":
        return lambda a, axis: np.nanmean(a, axis=axis)
    raise ValueError("Unknown agg. Use median or mean.")

def build_ensemble_basins_first(
    predictions: Dict[str, Dict[str, Dict[str, Any]]],
    variable_hint: Optional[str] = None,
    agg: str = "median",
    out_name: str = "Ensemble(mm/d)_sim",
    keep: str = "all",
    verbose: bool = True,
) -> Tuple[Dict[str, Dict[str, Dict[str, xr.DataArray]]], Dict[str, Any]]:
    agg_fn = robust_agg(agg)
    ensemble_result = {}
    report = {"seeds_total": len(predictions), "basins_total": 0, "basin_seed_built": 0,
              "skipped_no_data": 0, "skipped_shape_mismatch": 0, "shape_mismatch_examples": [],
              "other_errors": 0, "other_error_examples": []}

    for seed, basin_dict in predictions.items():
        report["basins_total"] += len(basin_dict)
        for basin, series_dict in basin_dict.items():
            sims, ref_da, ref_shape = [], None, None
            try:
                for series_key, sim_data in series_dict.items():
                    if variable_hint is not None and series_key != variable_hint:
                        continue
                    items = sim_data.items() if isinstance(sim_data, dict) else [("data", sim_data)]
                    for member_id, da in items:
                        if not isinstance(da, xr.DataArray):
                            continue
                        da = squeeze_prediction_da(da)
                        if ref_da is None:
                            ref_da, ref_shape = da, da.values.shape
                        if da.values.shape != ref_shape:
                            if keep == "strict":
                                raise ValueError(f"shape mismatch: {da.values.shape} != {ref_shape}")
                            report["skipped_shape_mismatch"] += 1
                            if len(report["shape_mismatch_examples"]) < 10:
                                report["shape_mismatch_examples"].append({"seed": seed, "basin": basin, "member": member_id})
                            continue
                        sims.append(da.values)
                if not sims or ref_da is None:
                    report["skipped_no_data"] += 1
                    continue
                stack = np.stack(sims, axis=0)
                ens = agg_fn(stack, 0)
                ensemble_da = xr.DataArray(ens, dims=ref_da.dims, coords=ref_da.coords,
                                           name=out_name, attrs=dict(ref_da.attrs) if ref_da.attrs else {})
                ensemble_da.attrs.update({"ensemble_agg": agg, "n_members": int(stack.shape[0])})
                ensemble_result.setdefault(str(basin), {})[str(seed)] = {"ensemble": ensemble_da}
                report["basin_seed_built"] += 1
            except Exception as e:
                report["other_errors"] += 1
                if len(report["other_error_examples"]) < 10:
                    report["other_error_examples"].append({"seed": seed, "basin": basin, "error": repr(e)})

    if verbose:
        print(report)
    return ensemble_result, report

# all_top10 = load_pickle(PICKLE_DIR / "All_Top10_Configs_CAMELSUS_test.p")
# all_cluster = load_pickle(PICKLE_DIR / "All_Cluster_wise_Configs_CAMELSUS_test.p")
# ens_top10, rep_top10 = build_ensemble_basins_first(all_top10, variable_hint=SIM_KEY, agg="median")
# ens_cluster, rep_cluster = build_ensemble_basins_first(all_cluster, variable_hint=SIM_KEY, agg="median")
# save_pickle(ens_top10, PICKLE_DIR / "ensemble_Top10_CAMELSUS_test.p")
# save_pickle(rep_top10, PICKLE_DIR / "report_ensemble_Top10_CAMELSUS_test.p")
# save_pickle(ens_cluster, PICKLE_DIR / "ensemble_Cluster_wise_CAMELSUS_test.p")
# save_pickle(rep_cluster, PICKLE_DIR / "report_ensemble_Cluster_wise_CAMELSUS_test.p")

## 12. Evaluate ensembles on test data

In [ ]:
def import_metrics_module(metrics_py: Path):
    import importlib.util
    import sys
    metrics_py = metrics_py.resolve()
    if not metrics_py.exists():
        raise FileNotFoundError(f"metrics.py not found: {metrics_py}")
    spec = importlib.util.spec_from_file_location("metrics", metrics_py)
    module = importlib.util.module_from_spec(spec)
    sys.modules["metrics"] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

def align_obs_sim(obs_da: xr.DataArray, sim_da: xr.DataArray) -> Tuple[xr.DataArray, xr.DataArray]:
    obs_da = squeeze_prediction_da(obs_da)
    sim_da = squeeze_prediction_da(sim_da)
    try:
        return xr.align(obs_da, sim_da, join="inner")
    except Exception:
        n = min(obs_da.size, sim_da.size)
        return obs_da.isel({obs_da.dims[0]: slice(0, n)}), sim_da.isel({sim_da.dims[0]: slice(0, n)})

def evaluate_model_dict(
    model_dict: Mapping[str, Mapping[str, Mapping[str, xr.DataArray]]],
    obs_dict: Mapping[str, xr.DataArray],
    model_name: str,
    metrics_module,
    sim_member_key: str = "ensemble",
    metric_names: Sequence[str] = ("NSE", "KGE", "MSE", "RMSE", "Alpha-NSE", "Beta-NSE", "Beta-KGE",
                                   "Pearson-r", "FHV", "FMS", "FLV", "Peak-MAPE", "MAPE"),
) -> pd.DataFrame:
    rows = []
    for basin, seed_dict in model_dict.items():
        if basin not in obs_dict:
            continue
        for seed, pred_dict in seed_dict.items():
            if sim_member_key not in pred_dict:
                continue
            obs_da, sim_da = align_obs_sim(obs_dict[basin], pred_dict[sim_member_key])
            try:
                vals = metrics_module.calculate_metrics(
                    obs_da, sim_da, metrics=list(metric_names), resolution="1D", datetime_coord=obs_da.dims[0]
                )
            except Exception as e:
                LOGGER.warning("Metric failure model=%s basin=%s seed=%s: %r", model_name, basin, seed, e)
                vals = {m: np.nan for m in metric_names}
            rows.append({"model": model_name, "basin": basin, "seed": seed, **vals})
    return pd.DataFrame(rows)

def evaluate_ensembles_to_csv(
    top10_pickle: Path,
    cluster_pickle: Path,
    obs_pickle: Path,
    metrics_py: Path,
    out_csv: Path,
) -> pd.DataFrame:
    metrics_module = import_metrics_module(metrics_py)
    top10 = load_pickle(top10_pickle)
    cluster = load_pickle(cluster_pickle)
    obs = load_pickle(obs_pickle)

    df = pd.concat([
        evaluate_model_dict(top10, obs, "Top10", metrics_module),
        evaluate_model_dict(cluster, obs, "Cluster-wise", metrics_module),
    ], ignore_index=True)

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False)

    summary = df.groupby("model").agg(
        n_rows=("NSE", "size"),
        median_NSE=("NSE", "median"),
        mean_NSE=("NSE", "mean"),
        median_KGE=("KGE", "median"),
        mean_KGE=("KGE", "mean"),
        q05_NSE=("NSE", lambda x: x.quantile(0.05)),
        q95_NSE=("NSE", lambda x: x.quantile(0.95)),
    ).reset_index()
    summary.to_csv(out_csv.with_name(out_csv.stem + "_summary.csv"), index=False)
    display(summary)
    return df

# test_metrics = evaluate_ensembles_to_csv(
#     PICKLE_DIR / "ensemble_Top10_CAMELSUS_test.p",
#     PICKLE_DIR / "ensemble_Cluster_wise_CAMELSUS_test.p",
#     PICKLE_DIR / "Obs_test.p",
#     PROJECT_ROOT / "metrics.py",
#     EVAL_DIR / "test_metrics_ensembles_long.csv",
# )

## 13. Minimal execution order

### Full reproduction

1. Edit **User configuration**.
2. Generate random-search YAML configs.
3. Run HPO training.
4. Extract validation metrics.
5. Select Top-10 and cluster-wise configs.
6. Create final seeded configs.
7. Train final selected configs.
8. Evaluate final selected runs on test.
9. Extract test predictions and observations.
10. Build median ensembles.
11. Evaluate ensembles.

### Faster route

- If HPO runs already exist, start from validation extraction.
- If final selected runs already exist and contain `test_results.p`, start from prediction extraction.
- If prediction pickles already exist, start from ensemble construction.
- If ensemble pickles already exist, start from final test metrics.